## 1. Install and import libraries

In [ ]:
!pip -q install kagglehub opencv-python-headless scikit-learn tqdm

import os
import cv2
import math
import json
import random
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2. Dataset and frame paths

In [ ]:
# If you already extracted frames for ResNet50 / EfficientNet, use that SAME folder.
# Typical Google Colab path from the previous experiments:
FRAMES_DIR = Path("/content/celebdf_frames")

# KaggleHub source (used only if frame extraction is needed)
USE_KAGGLEHUB_IF_FRAMES_MISSING = True
DATASET_HANDLE = "reubensuju/celeb-df-v2"

# Frame extraction settings — keep these identical to the other model experiments.
FRAMES_PER_VIDEO = 5
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

print("Frames directory:", FRAMES_DIR)
print("Already extracted:", FRAMES_DIR.exists())


## 3. (Optional) Download dataset and extract frames

In [ ]:
# This cell runs only if FRAMES_DIR does not already contain train/val/test folders.
# It uses the official testing-video list when available. If your existing manifests
# are present, they are used to preserve the same train/val/test split.

def find_video_root(root):
    root = Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for p in candidates:
        names = {x.name for x in p.iterdir()} if p.exists() else set()
        if {"YouTube-real", "Celeb-real", "Celeb-synthesis"}.issubset(names):
            return p
    raise FileNotFoundError("Could not find YouTube-real, Celeb-real and Celeb-synthesis folders.")

def get_video_records(dataset_root):
    dataset_root = Path(dataset_root)
    video_root = find_video_root(dataset_root)

    # Search for the official test list.
    test_list = None
    for p in dataset_root.rglob("List_of_testing_videos.txt"):
        test_list = p
        break
    if test_list is None:
        for p in video_root.parent.rglob("List_of_testing_videos.txt"):
            test_list = p
            break

    # Each line of List_of_testing_videos.txt looks like "1 YouTube-real/00170.mp4"
    # (a label, a space, then the path). The label must be stripped off, otherwise
    # nothing ever matches and the test split ends up empty.
    def make_key(p):
        return "/".join(str(p).strip().replace("\\", "/").split("/")[-2:])   # "folder/video.mp4"

    test_keys = set()
    if test_list:
        for line in test_list.read_text(errors="ignore").splitlines():
            parts = line.strip().split(maxsplit=1)
            if parts:
                test_keys.add(make_key(parts[-1]))
    print("Test list file:", test_list)
    print("Test-list entries:", len(test_keys))

    records = []
    for folder, label in [("YouTube-real", 0), ("Celeb-real", 0), ("Celeb-synthesis", 1)]:
        for video in (video_root / folder).rglob("*"):
            if video.is_file() and video.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}:
                rel = video.relative_to(video_root).as_posix()
                is_test = make_key(rel) in test_keys
                records.append({"path": str(video), "relative_path": rel, "label": label, "split": "test" if is_test else "trainval"})
    records = pd.DataFrame(records)
    matched = int((records["split"] == "test").sum())
    print("Official test videos matched:", matched, "of", len(test_keys))
    if matched == 0:
        raise RuntimeError("0 test videos matched - check the test list format. Stopping so no leaky split is built.")
    if matched != len(test_keys):
        print("WARNING: some test-list videos were not found in the dataset folders.")
    return records, video_root

def extract_uniform_frames(video_path, output_dir, label_name, count=5, size=224):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or total <= 0:
        cap.release()
        return 0
    indices = np.linspace(0, max(total - 1, 0), count, dtype=int)
    saved = 0
    output_dir = Path(output_dir) / label_name
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(video_path).stem
    for i, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        out = output_dir / f"{stem}_frame_{i}.jpg"
        cv2.imwrite(str(out), cv2.cvtColor(frame, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])
        saved += 1
    cap.release()
    return saved

if not all((FRAMES_DIR / s).exists() for s in ["train", "val", "test"]):
    if not USE_KAGGLEHUB_IF_FRAMES_MISSING:
        raise FileNotFoundError("Set FRAMES_DIR to your existing extracted frame directory.")
    import kagglehub
    downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
    print("Downloaded dataset:", downloaded)

    records, video_root = get_video_records(downloaded)
    print("Videos found:", len(records), "| official test videos:", (records.split == "test").sum())

    # Split non-test videos into train/validation with the same 15% stratified split convention.
    from sklearn.model_selection import train_test_split
    trainval = records[records.split == "trainval"].copy()
    train_df, val_df = train_test_split(
        trainval, test_size=0.15, random_state=SEED, stratify=trainval["label"]
    )
    split_frames = {
        "train": train_df,
        "val": val_df,
        "test": records[records.split == "test"].copy()
    }

    # Remove old/partial frames first. Frames from an earlier run that built a wrong
    # (test-less) split would leave test videos inside train/val = data leakage.
    shutil.rmtree(FRAMES_DIR, ignore_errors=True)
    FRAMES_DIR.mkdir(parents=True, exist_ok=True)
    for split_name, df in split_frames.items():
        print(f"Extracting {split_name}: {len(df)} videos")
        for row in tqdm(df.itertuples(index=False), total=len(df)):
            class_name = "FAKE" if row.label == 1 else "REAL"
            extract_uniform_frames(row.path, FRAMES_DIR / split_name, class_name, FRAMES_PER_VIDEO, IMAGE_SIZE)
else:
    print("Using existing extracted frames. No download or extraction required.")


## 4. Verify class counts and create DataLoaders

In [ ]:
# ImageFolder sorts class names alphabetically: FAKE=0, REAL=1.
# This is the expected mapping used in the previous frame-based experiments.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(FRAMES_DIR / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(FRAMES_DIR / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(FRAMES_DIR / "test", transform=eval_transform)

print("Class mapping:", train_dataset.class_to_idx)
print("Train:", len(train_dataset), "Validation:", len(val_dataset), "Test:", len(test_dataset))
assert train_dataset.class_to_idx == {"FAKE": 0, "REAL": 1}, "Unexpected class mapping."
assert val_dataset.class_to_idx == train_dataset.class_to_idx
assert test_dataset.class_to_idx == train_dataset.class_to_idx

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape, "| labels:", labels.shape)


## 5. Define the custom CNN

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ]
        if pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class DeepfakeCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.45):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = DeepfakeCNN().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 6. Loss, optimizer and checkpoint setup

In [ ]:
# Class-weighted loss helps when the REAL and FAKE frame counts are imbalanced.
counts = np.bincount([label for _, label in train_dataset.samples], minlength=2)
class_weights = len(train_dataset) / (2.0 * np.maximum(counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

EPOCHS = 15
BEST_MODEL_PATH = "cnn_deepfake_best.pth"
CHECKPOINT_PATH = "cnn_checkpoint.pth"

print("Class counts [FAKE, REAL]:", counts)
print("Class weights:", class_weights.detach().cpu().numpy())


## 7. Training and validation functions

In [ ]:
def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss = 0.0
    all_labels, all_preds = [], []

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            all_labels.extend(labels.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
start_epoch = 0


## 8. Train the CNN (with resumable checkpoints)

In [ ]:
# To resume after an interruption, set RESUME_TRAINING = True.
RESUME_TRAINING = False

if RESUME_TRAINING and os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    history = checkpoint.get("history", history)
    print("Resuming from epoch", start_epoch + 1)

for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)
    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f}, acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_to_idx": train_dataset.class_to_idx,
            "image_size": IMAGE_SIZE,
            "best_val_acc": best_val_acc
        }, BEST_MODEL_PATH)
        print("Saved best model.")

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "best_val_acc": best_val_acc,
        "history": history
    }, CHECKPOINT_PATH)

print("Best validation accuracy:", best_val_acc)
print("Best model:", BEST_MODEL_PATH)


## 9. Plot training history

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_ran, history["train_loss"], label="Train")
plt.plot(epochs_ran, history["val_loss"], label="Validation")
plt.title("CNN Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_ran, history["train_acc"], label="Train")
plt.plot(epochs_ran, history["val_acc"], label="Validation")
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.legend()
plt.tight_layout()
plt.show()


## 10. Evaluate the best model on the held-out test set

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_labels, all_probs, all_preds = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)[:, 1]  # probability of REAL (class 1)
        preds = (probs >= 0.5).long()

        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)

print("TEST CLASSIFICATION REPORT")
print(classification_report(all_labels, all_preds, target_names=["FAKE", "REAL"], digits=4))
print("Test accuracy:", accuracy_score(all_labels, all_preds))
try:
    print("ROC-AUC:", roc_auc_score(all_labels, all_probs))
except ValueError:
    print("ROC-AUC could not be calculated (only one class in test set).")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("CNN Confusion Matrix")
plt.xticks([0, 1], ["FAKE", "REAL"])
plt.yticks([0, 1], ["FAKE", "REAL"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")
plt.tight_layout()
plt.show()


### Confusion matrix (counts and row-normalized)

In [ ]:
# Uses all_labels / all_preds from the test evaluation cell above.
# Class mapping: FAKE = 0, REAL = 1
class_names = ["FAKE", "REAL"]
cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # each row sums to 1 (recall per class)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Left: raw counts
im0 = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("CNN Confusion Matrix (counts)")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
thresh0 = cm.max() / 2
for (i, j), v in np.ndenumerate(cm):
    axes[0].text(j, i, f"{v}", ha="center", va="center",
                 color="white" if v > thresh0 else "black", fontsize=13)

# Right: row-normalized (% of each actual class)
im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("CNN Confusion Matrix (row-normalized)")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
for (i, j), v in np.ndenumerate(cm_norm):
    axes[1].text(j, i, f"{v*100:.1f}%", ha="center", va="center",
                 color="white" if v > 0.5 else "black", fontsize=13)

for ax in axes:
    ax.set_xticks([0, 1]); ax.set_xticklabels(class_names)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

# Breakdown (FAKE is treated as the "positive" class for detection)
tp, fn = cm[0, 0], cm[0, 1]   # FAKE correctly caught / FAKE missed (called REAL)
fp, tn = cm[1, 0], cm[1, 1]   # REAL wrongly flagged as FAKE / REAL correctly passed
print(f"FAKE caught (TP):          {tp}")
print(f"FAKE missed as REAL (FN):  {fn}")
print(f"REAL flagged as FAKE (FP): {fp}")
print(f"REAL correct (TN):         {tn}")
print(f"Overall accuracy:          {(tp + tn) / cm.sum():.4f}")


## 11. Optional: choose a different probability threshold

In [ ]:
# The default threshold is 0.50. This cell lets you inspect how other thresholds
# change precision/recall. Choose the threshold using validation data in a final study,
# not by repeatedly tuning on the test set.
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    threshold_preds = (all_probs >= threshold).astype(int)
    print(f"\nThreshold: {threshold:.2f}")
    print(classification_report(
        all_labels, threshold_preds, target_names=["FAKE", "REAL"], digits=4
    ))


## 12. Save a clean state-dict file and download

In [ ]:
# Save a simple .pth state dictionary as well, for backend integration.
STATE_DICT_PATH = "cnn_deepfake_state_dict.pth"
torch.save(model.state_dict(), STATE_DICT_PATH)

print("Saved files:")
print(BEST_MODEL_PATH)
print(CHECKPOINT_PATH)
print(STATE_DICT_PATH)

# In Google Colab, uncomment to download the best model:
# from google.colab import files
# files.download(BEST_MODEL_PATH)
# files.download(STATE_DICT_PATH)
